# 01 - Data Understanding

## Objective

The objective of this notebook is to perform an initial inspection of the FAA Wildlife Strike dataset before any preprocessing.

The notebook focuses on:

- Understanding the dataset structure
- Inspecting data quality
- Identifying missing values
- Detecting duplicate records
- Understanding variable types
- Recording observations for the data preparation stage


## 1. Import Libraries

This section imports the libraries required for the initial inspection of the FAA Wildlife Strike dataset.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from pathlib import Path

## 2. Define Project Paths

The notebook is located inside the `notebooks` folder. Therefore, the project root is one directory above the current notebook location.

In [3]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw-data directory: {RAW_DATA_DIR}")
print(f"Processed-data directory: {PROCESSED_DATA_DIR}")

Project root: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis
Raw-data directory: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\raw
Processed-data directory: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\processed


## 3. Locate the Raw Dataset

Before reading the dataset, we confirm which files are available in the raw-data directory.

In [4]:
raw_files = list(RAW_DATA_DIR.glob("*"))

if not raw_files:
    print("No raw-data file was found. Add the FAA dataset to data/raw/.")
else:
    for file_path in raw_files:
        print(file_path.name)

.gitkeep
faa_strikes.csv


## 4. Load the Dataset

The dataset is loaded without applying any cleaning so that the original structure can be examined first.

In [6]:
DATA_FILE = RAW_DATA_DIR / "faa_strikes.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_FILE}\n"
        "Copy the FAA CSV file into the data/raw folder."
    )

df = pd.read_csv(
    DATA_FILE,
    encoding="latin-1",
    low_memory=False
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 5. Dataset Dimensions

The number of rows represents reported wildlife-strike records, while the number of columns represents the available variables.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.info()
df.describe(include="all").T

Rows: 348,146
Columns: 103


## 6. Sample Records

A small sample is displayed to understand the structure and formatting of the original records.

In [8]:
df.head()

,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,AIRPORT_LATITUDE,AIRPORT_LONGITUDE,...,NR_INJURIES,NR_FATALITIES,COMMENTS,IMAGE,REPORTED_NAME,REPORTED_TITLE,SOURCE,PERSON,LUPDATE,TRANSFER
0,608242,6/22/1996 0:00:00,6,1996,NaN,NaN,KSMF,SACRAMENTO INTL,NaN,NaN,...,NaN,NaN,/Legacy Record 100001/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
1,608243,6/26/1996 0:00:00,6,1996,NaN,NaN,KDEN,DENVER INTL AIRPORT,NaN,NaN,...,NaN,NaN,/Legacy Record 100002/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
2,608244,7/1/1996 0:00:00,7,1996,NaN,NaN,KOMA,EPPLEY AIRFIELD,NaN,NaN,...,NaN,NaN,/Legacy Record 100003/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
3,608245,7/1/1996 0:00:00,7,1996,NaN,NaN,KIAD,WASHINGTON DULLES INTL ARPT,NaN,NaN,...,NaN,NaN,/Legacy Record 100004/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0
4,608246,7/1/1996 0:00:00,7,1996,NaN,NaN,KLGA,LA GUARDIA ARPT,NaN,NaN,...,NaN,NaN,/Legacy Record 100005/,0,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,12/20/2007 0:00:00,0


## 7. Columns and Data Types

This inspection identifies numeric, text, date-like, and categorical variables that may require different preparation steps.

In [9]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "unique_values": df.nunique(dropna=True).values,
})

column_summary

,column,dtype,non_null_count,unique_values
0,INDEX_NR,int64,348146,348139
1,INCIDENT_DATE,object,348146,13223
2,INCIDENT_MONTH,int64,348146,12
3,INCIDENT_YEAR,int64,348146,37
4,TIME,object,226783,1441
...,...,...,...,...
98,REPORTED_TITLE,object,348146,1
99,SOURCE,object,348146,16
100,PERSON,object,327040,6
101,LUPDATE,object,348146,4834


## 8. Missing-Value Overview

Missingness is measured before cleaning to identify variables that may require exclusion, imputation, grouping, or special interpretation.

In [10]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(
          missing_percent=lambda x:
              (x["missing_count"] / len(df) * 100).round(2)
      )
      .sort_values("missing_percent", ascending=False)
)

missing_summary.head(20)

,missing_count,missing_percent
INCIDENT_LATITUDE,348146,100.00
INCIDENT_LONGITUDE,348146,100.00
AIRPORT_LATITUDE,348146,100.00
AIRPORT_LONGITUDE,348146,100.00
NR_FATALITIES,348121,99.99
NR_INJURIES,347844,99.91
BIRD_BAND_NUMBER,347352,99.77
EFFECT_OTHER,345363,99.20
ENG_4_POS,344505,98.95
COST_REPAIRS,342721,98.44


## 9. Exact Duplicate Records

Exact duplicates are inspected but are not removed at this stage because this notebook focuses on understanding the original dataset.

In [11]:
exact_duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{exact_duplicate_count / len(df) * 100:.2f}%"
)

Exact duplicate rows: 0
Duplicate percentage: 0.00%


## 10. Initial Observations

### Summary

Based on the initial inspection:

- The dataset was successfully loaded.
- Missing values exist in several variables.
- Duplicate records will be examined further during data preparation.
- Variables include numerical, categorical, and date fields.
- Data cleaning is required before modelling.

The next notebook (02_data_preparation.ipynb) will perform cleaning and preprocessing.